In [ ]:
import numpy as np
import pandas as pd
#import sys
#import os

# PSG environment
#sys.path.insert(1, os.path.join(os.environ.get('PSG25_HOME'), 'Python'))
#from psg_preload_libraries import initialize_psg_environment
#initialize_psg_environment()
#import psgpython as psg
from matplotlib import pyplot as plt

# Read data
# data = pd.read_csv(r"C:\Users\niewu\OneDrive - Stony Brook University\Desktop\problem_st_dev_risk_Run_data\matrix_scenarios.txt",
#                     delim_whitespace=True) 

data = pd.read_csv(r"C:\Users\JUANJO\Downloads\problem_st_dev_risk_Run_data\matrix_scenarios.txt",
                    delim_whitespace=True) 

# Base problem statement template
# We will replace {MATRIX_NAME} later
problem_statement_template = """
for {matrix_fact_in; matrix_fact_out; num} = crossvalidation(5, {MATRIX_NAME})
minimize
cvar_dev(0.95,matrix_fact_in)
Value:
L_num(matrix_fact_out)
L_num(matrix_fact_in)
cvar_dev(0.9, matrix_fact_in)
meanabs_dev(matrix_fact_in)
st_dev(matrix_fact_in)
var_risk(0.75, matrix_fact_in)
var_risk_g(0.75, matrix_fact_in)
var_risk(0.9, matrix_fact_in)
var_risk_g(0.9, matrix_fact_in)
end for
"""

# Problem statement for out-of-sample
problem_statement_out_of_sample = """
minimize
cvar_dev(0.95, matrix_s)
"""

# PSG format

# 3-Factor Data
header_3fact = ['x1', 'x2', 'x3', 'scenario_benchmark']
matrix_scenarios_3fact = data[header_3fact].values

# 2-Factor Data (x1 and x3, Last name: Wu)
header_2fact = ['x1', 'x3', 'scenario_benchmark']
matrix_scenarios_2fact = data[header_2fact].values

# Set up Simulation Parameters

# We start with 10 since starting with 3 as wanted yields an error for 
# the PSG solver since we want to do 5-fold but in that case we will 
# only have 3 rows of data

# Create a sequence of training sizes
training_sizes = [10] + list(range(20, 901, 20))

# Number of times we are going to run each simulation
n_repetitions = 25  # Minimum is 20, we can select more

# Total number of rows
n_total_rows = matrix_scenarios_3fact.shape[0]

# Testing set will always contain 100 rows
n_test = 100

all_indices = np.arange(n_total_rows)

# Initialize Results Storage

# We will store results in a list of dictionaries,
# then convert to a DataFrame at the end
results_list = []

# Outer loop: to iterate over each training size
for n_train in training_sizes:
    
    print(f"Starting training size: {n_train}")
    
    # Temporary matrices to store results from each rep
    # We initialize with NaNs
    in_sample_2f_reps = np.full((n_repetitions, 2), np.nan)
    in_sample_3f_reps = np.full((n_repetitions, 3), np.nan)
    out_sample_2f_reps = np.full((n_repetitions, 2), np.nan)
    out_sample_3f_reps = np.full((n_repetitions, 3), np.nan)
    
    # Inner loop: Repeat the experiment n_repetitions times
    for i in range(n_repetitions):
        
        # Step I: Randomly select 100 rows for testing set
        test_indices = np.random.choice(all_indices, n_test, replace=False)
        remaining_indices = np.setdiff1d(all_indices, test_indices)
        
        # Step II: Randomly select n_train rows from remaining
        train_indices = np.random.choice(remaining_indices, n_train, replace=False)
        
        # Create the training and testing matrices for this iteration
        matrix_train_set_3f = matrix_scenarios_3fact[train_indices]
        matrix_test_set_3f = matrix_scenarios_3fact[test_indices]
        
        matrix_train_set_2f = matrix_scenarios_2fact[train_indices]
        matrix_test_set_2f = matrix_scenarios_2fact[test_indices]
        
        # Step III: Fit models and get In-Sample CVaR Deviation
        
        # 2-Factor In-Sample
        problem_dict_in_2f = {
            'problem_name': f'cvar_in_2f_train_{n_train}_rep_{i}',
            'problem_statement': problem_statement_template.replace(
                "{MATRIX_NAME}", "matrix_train_set_2f"
            ),
            'matrix_train_set_2f': [header_2fact, matrix_train_set_2f]
        }
        output_in_2f = psg_solver(problem_dict_in_2f, True, False)
        
        # --- [USER TO FILL IN] ---
        # You need to fill in this logic based on how `output_in_2f` is structured.
        # `output_in_2f` is a list of 5 dictionaries (one for each fold).
        # We assume you extract the average value for 'x1' and 'x3'.
        def extract_in_sample_2f(output):
            # Example:
            x1_vals = [fold['point_problem_1']['x1'] for fold in output]
            x3_vals = [fold['point_problem_1']['x3'] for fold in output]
            avg_x1 = np.mean(x1_vals)
            avg_x3 = np.mean(x3_vals)
            return [avg_x1, avg_x3]
        
        in_sample_2f_reps[i, :] = extract_in_sample_2f(output_in_2f)
        # --- [END USER SECTION] ---

        
        # 3-Factor In-Sample
        problem_dict_in_3f = {
            'problem_name': f'cvar_in_3f_train_{n_train}_rep_{i}',
            'problem_statement': problem_statement_template.replace(
                "{MATRIX_NAME}", "matrix_train_set_3f"
            ),
            'matrix_train_set_3f': [header_3fact, matrix_train_set_3f]
        }
        output_in_3f = psg_solver(problem_dict_in_3f, True, False)
        
        # --- [USER TO FILL IN] ---
        # You need to fill in this logic for the 3-factor model.
        # We assume you extract the average value for 'x1', 'x2', and 'x3'.
        def extract_in_sample_3f(output):
            # Example:
            x1_vals = [fold['point_problem_1']['x1'] for fold in output]
            x2_vals = [fold['point_problem_1']['x2'] for fold in output]
            x3_vals = [fold['point_problem_1']['x3'] for fold in output]
            avg_x1 = np.mean(x1_vals)
            avg_x2 = np.mean(x2_vals)
            avg_x3 = np.mean(x3_vals)
            return [avg_x1, avg_x2, avg_x3]
            
        in_sample_3f_reps[i, :] = extract_in_sample_3f(output_in_3f)
        # --- [END USER SECTION] ---

        
        # --- Step IV: Calculate Out-Of-Sample CVaR Deviation ---
        
        # 2-Factor Out-of-Sample
        n_rows_2f = matrix_test_set_2f.shape[0]
        intercept_2f = np.ones((n_rows_2f, 1))
        
        # Build the new matrix in the required order:
        # [intercept, scenario_benchmark, x1, x3]
        # Original 2f matrix columns: [x1 (0), x3 (1), scenario_benchmark (2)]
        matrix_s_2f_data = np.hstack([
            intercept_2f,
            matrix_test_set_2f[:, [2]], # scenario_benchmark
            matrix_test_set_2f[:, [0]], # x1
            matrix_test_set_2f[:, [1]]  # x3
        ])
        header_s_2f = ['intercept', 'scenario_benchmark', 'x1', 'x3']
        
        problem_dict_out_2f = {
            'problem_name': 'cvar_out_2f',
            'problem_statement': problem_statement_out_of_sample,
            'matrix_s': [header_s_2f, matrix_s_2f_data]
        }
        output_out_2f = psg_solver(problem_dict_out_2f, True, False)
        
        # Extract results based on your logic:
        # sol = [['intercept', 'x1', 'x3'], array([val_int, val_sb, val_x1, val_x3])]
        # We want the *last 2* elements of the array.
        sol_vector_2f = output_out_2f['point_problem_1'][1]
        out_sample_2f_reps[i, :] = sol_vector_2f[-2:] # Gets [val_x1, val_x3]
        
        
        # 3-Factor Out-of-Sample
        n_rows_3f = matrix_test_set_3f.shape[0]
        intercept_3f = np.ones((n_rows_3f, 1))
        
        # Build the new matrix in the required order:
        # [intercept, scenario_benchmark, x1, x2, x3]
        # Original 3f matrix columns: [x1 (0), x2 (1), x3 (2), scenario_benchmark (3)]
        matrix_s_3f_data = np.hstack([
            intercept_3f,
            matrix_test_set_3f[:, [3]], # scenario_benchmark
            matrix_test_set_3f[:, [0]], # x1
            matrix_test_set_3f[:, [1]], # x2
            matrix_test_set_3f[:, [2]]  # x3
        ])
        header_s_3f = ['intercept', 'scenario_benchmark', 'x1', 'x2', 'x3']
        
        problem_dict_out_3f = {
            'problem_name': 'cvar_out_3f',
            'problem_statement': problem_statement_out_of_sample,
            'matrix_s': [header_s_3f, matrix_s_3f_data]
        }
        output_out_3f = psg_solver(problem_dict_out_3f, True, False)
        
        # Extract results:
        # sol = [['intercept', ...], array([val_int, val_sb, val_x1, val_x2, val_x3])]
        # We want the *last 3* elements of the array.
        sol_vector_3f = output_out_3f['point_problem_1'][1]
        out_sample_3f_reps[i, :] = sol_vector_3f[-3:] # Gets [val_x1, val_x2, val_x3]
        
    # End of inner loop (n_repetitions)
    
    # --- Step V: Calculate Averages ---
    # `np.nanmean` is the equivalent of R's `colMeans(..., na.rm = TRUE)`
    avg_in_2f = np.nanmean(in_sample_2f_reps, axis=0)
    avg_in_3f = np.nanmean(in_sample_3f_reps, axis=0)
    avg_out_2f = np.nanmean(out_sample_2f_reps, axis=0)
    avg_out_3f = np.nanmean(out_sample_3f_reps, axis=0)
    
    # --- Step VI: Store Averages ---
    results_list.append({
        'TrainingSize': n_train,
        # 2-Factor (x1, x3)
        'Avg_InSample_2F_f1': avg_in_2f[0], # Corresponds to x1
        'Avg_InSample_2F_f2': avg_in_2f[1], # Corresponds to x3
        
        'Avg_OutOfSample_2F_f1': avg_out_2f[0], # Corresponds to x1
        'Avg_OutOfSample_2F_f2': avg_out_2f[1], # Corresponds to x3
        
        # 3-Factor (x1, x2, x3)
        'Avg_InSample_3F_f1': avg_in_3f[0], # Corresponds to x1
        'Avg_InSample_3F_f2': avg_in_3f[1], # Corresponds to x2
        'Avg_InSample_3F_f3': avg_in_3f[2], # Corresponds to x3
        
        'Avg_OutOfSample_3F_f1': avg_out_3f[0], # Corresponds to x1
        'Avg_OutOfSample_3F_f2': avg_out_3f[1], # Corresponds to x2
        'Avg_OutOfSample_3F_f3': avg_out_3f[2], # Corresponds to x3
    })
    
# End of outer loop (training_sizes)

# --- 7. Final Results ---
results_df = pd.DataFrame(results_list)

print("Simulation Complete. Final averaged results:")
print(results_df)

# You can save this to a CSV:
# results_df.to_csv("simulation_results.csv", index=False)

C:\Users\JUANJO\AppData\Local\Temp\ipykernel_15112\1824460681.py:17: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  data = pd.read_csv(r"C:\Users\JUANJO\Downloads\problem_st_dev_risk_Run_data\matrix_scenarios.txt",
